Mount Drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


Imports

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

import shutil
import os

import matplotlib.pyplot as plt

Define Paths

In [3]:
BASE_PATH = Path("/content/drive/MyDrive/fyp")


DATASET_PATH = BASE_PATH / "Processed Dataset"


IMAGE_DIR = DATASET_PATH / "images"

MASK_DIR = DATASET_PATH / "masks"


METADATA_PATH = DATASET_PATH / "metadata.csv"


OUTPUT_PATH = BASE_PATH / "Federated_Data"


print(METADATA_PATH)

/content/drive/MyDrive/fyp/Processed Dataset/metadata.csv


In [4]:
df = pd.read_csv(
    METADATA_PATH
)


df.head()

,image,mask,class,label,patient_id,source,client_id
0,cheng_000000.png,cheng_000000_mask.png,meningioma,1,100360,cheng,NaN
1,cheng_000001.png,cheng_000001_mask.png,meningioma,1,101016,cheng,NaN
2,cheng_000002.png,cheng_000002_mask.png,meningioma,1,107494,cheng,NaN
3,cheng_000003.png,cheng_000003_mask.png,pituitary,3,112649,cheng,NaN
4,cheng_000004.png,cheng_000004_mask.png,pituitary,3,112649,cheng,NaN


In [7]:
df.groupby('class')['class'].value_counts()

,count
class,
glioma,1426
meningioma,708
no_tumor,1414
pituitary,930


Handling non_tumor class patient Id for the sampling

In [8]:
no_tumor_mask = (
    df["source"] == "kaggle"
)


counter = 1


for idx in df[no_tumor_mask].index:

    df.loc[idx,"patient_id"] = (
        f"NT_{counter:06d}"
    )

    counter += 1

In [9]:
df[df["source"]=="kaggle"]["patient_id"].head()

,patient_id
3064,NT_000001
3065,NT_000002
3066,NT_000003
3067,NT_000004
3068,NT_000005


Unique pateint id in cheng data

In [10]:
cheng_patients = (
    df[df["source"]=="cheng"]
    ["patient_id"]
    .nunique()
)


print(
    "Cheng patients:",
    cheng_patients
)

#3064 images were taken  from 233 patients

Cheng patients: 233


In [11]:
df[
    df["source"]=="cheng"
]["patient_id"].value_counts().head()

#Patients with most MRI scans

,count
patient_id,
MR032322C,38
MR051644,37
MR051644C,35
MR050135B,34
MR040240C,31


Defining Hospitals

In [12]:
HOSPITALS = [
    "Hospital_A",
    "Hospital_B",
    "Hospital_C",
    "Hospital_D"
]


NUM_CLIENTS = len(HOSPITALS)

Sampling by IID Technique

In [13]:
def iid_split(dataframe,num_clients=4,seed=42):

    np.random.seed(seed)

    client_data = {
        i: []
        for i in range(num_clients)
    }


    for cls in dataframe["class"].unique():

        class_data = dataframe[
            dataframe["class"] == cls
        ].copy()


        patients = (
            class_data["patient_id"]
            .unique()
        )


        np.random.shuffle(
            patients
        )


        patient_chunks = np.array_split(
            patients,
            num_clients
        )


        for i,patients_chunk in enumerate(patient_chunks):

            client_rows = class_data[
                class_data["patient_id"]
                .isin(patients_chunk)
            ]


            client_data[i].append(
                client_rows
            )


    final_clients = {}


    for i in client_data:

        final_clients[i] = pd.concat(
            client_data[i],
            ignore_index=True
        )


    return final_clients

Generate IID-Clients

In [14]:
iid_clients = iid_split(
    df,
    NUM_CLIENTS
)


for i,data in iid_clients.items():

    print(
        HOSPITALS[i],
        len(data)
    )

Hospital_A 1224
Hospital_B 1157
Hospital_C 1006
Hospital_D 1091


Dirichlet Sampling Function for Non-IID Split

In [15]:
def dirichlet_split(
    dataframe,
    num_clients=4,
    alpha=0.5,
    seed=42
):

    np.random.seed(seed)


    client_indices = {
        i:[]
        for i in range(num_clients)
    }


    classes = dataframe["label"].unique()


    for cls in classes:

        class_indices = dataframe[
            dataframe["label"] == cls
        ].index.values


        np.random.shuffle(
            class_indices
        )


        proportions = np.random.dirichlet(
            np.repeat(alpha,num_clients)
        )


        splits = (
            proportions *
            len(class_indices)
        ).astype(int)


        splits[-1] = (
            len(class_indices)
            -
            splits[:-1].sum()
        )


        start = 0


        for client,amount in enumerate(splits):

            client_indices[client].extend(
                class_indices[
                    start:start+amount
                ]
            )

            start += amount



    clients={}


    for client,indices in client_indices.items():

        clients[client] = dataframe.loc[
            indices
        ].reset_index(drop=True)


    return clients

Generate Non-IID Clients

In [16]:
noniid_clients = dirichlet_split(
    df,
    num_clients=4,
    alpha=0.5
)


for i,data in noniid_clients.items():

    print(
        HOSPITALS[i],
        len(data)
    )

Hospital_A 554
Hospital_B 831
Hospital_C 1432
Hospital_D 1661


Check Distributions

For IID-Splt

In [17]:
for i,data in iid_clients.items():

    print("\n",HOSPITALS[i])

    print(
        data["class"]
        .value_counts()
    )


 Hospital_A
class
glioma        440
no_tumor      354
pituitary     229
meningioma    201
Name: count, dtype: int64

 Hospital_B
class
glioma        359
no_tumor      354
pituitary     261
meningioma    183
Name: count, dtype: int64

 Hospital_C
class
no_tumor      353
glioma        252
pituitary     236
meningioma    165
Name: count, dtype: int64

 Hospital_D
class
glioma        375
no_tumor      353
pituitary     204
meningioma    159
Name: count, dtype: int64


For non-IID Clients

In [18]:
for i,data in noniid_clients.items():

    print("\n",HOSPITALS[i])

    print(
        data["class"]
        .value_counts()
    )


 Hospital_A
class
no_tumor      419
pituitary      93
meningioma     35
glioma          7
Name: count, dtype: int64

 Hospital_B
class
no_tumor      480
meningioma    287
pituitary      54
glioma         10
Name: count, dtype: int64

 Hospital_C
class
glioma        1062
pituitary      219
no_tumor        97
meningioma      54
Name: count, dtype: int64

 Hospital_D
class
pituitary     564
no_tumor      418
glioma        347
meningioma    332
Name: count, dtype: int64


Save Client Data

In [19]:
def save_clients(clients,folder_name):

    output = OUTPUT_PATH / folder_name


    for i,data in clients.items():

        hospital = output / HOSPITALS[i]

        hospital.mkdir(
            parents=True,
            exist_ok=True
        )


        data.to_csv(
            hospital/"metadata.csv",
            index=False
        )


save_clients(
    iid_clients,
    "IID"
)


save_clients(
    noniid_clients,
    "NON_IID_Dirichlet"
)